# 1장. Data Collection (Finger Classification)
이 챕터에서는 CNN 모델을 사용해서 사람의 손가락 편 개수(0 ~ 5)를 분류하는 데이터셋을 수집할 것입니다.
카메라 피드를 보며 0부터 5까지의 손가락 모양을 취하고 해당하는 클래스 폴더에 이미지를 저장합니다.

In [ ]:
import os
import shutil

# 기존 데이터셋 및 모델 초기화 (안전하게 삭제)
if os.path.exists('dataset_finger'):
    shutil.rmtree('dataset_finger')
    print('기존 dataset_finger 폴더 삭제 완료.')
else:
    print('dataset_finger 폴더가 없습니다.')

if os.path.exists('best_model_finger.pth'):
    os.remove('best_model_finger.pth')
    print('기존 best_model_finger.pth 파일 삭제 완료.')
else:
    print('best_model_finger.pth 파일이 없습니다.')

In [ ]:
import os
import uuid
import glob
import time
import ipywidgets as widgets
from IPython.display import display
import traitlets
from jetbot import Camera, bgr8_to_jpeg

DATASET_DIR = 'dataset_finger'
CLASSES = ['0', '1', '2', '3', '4', '5']

for c in CLASSES:
    os.makedirs(os.path.join(DATASET_DIR, c), exist_ok=True)

print('Dataset directories created successfully!')

## 카메라 및 UI 생성하기
카메라 송출 화면과 저장할 클래스를 선택할 수 있는 버튼, 그리고 저장 버튼을 표시합니다.

In [ ]:
camera = Camera()

image_widget = widgets.Image(format='jpeg', width=224, height=224)
traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

class_select = widgets.ToggleButtons(
    options=CLASSES,
    description='Class (Fingers):',
    disabled=False,
    button_style='info',
)

save_button = widgets.Button(description='Save Image', button_style='success')
count_label = widgets.HTML(value='')

def get_counts_html():
    counts = {}
    for c in CLASSES:
        path = os.path.join(DATASET_DIR, c, '*.jpg')
        counts[c] = len(glob.glob(path))
    html = '<h4>Collected Images:</h4>'
    for c in CLASSES:
        html += f'<b>Class {c}:</b> {counts[c]} images<br>'
    return html

def update_counts():
    count_label.value = get_counts_html()

def save_snapshot(change):
    selected_class = class_select.value
    filename = f'{selected_class}_{str(uuid.uuid1())}.jpg'
    filepath = os.path.join(DATASET_DIR, selected_class, filename)
    with open(filepath, 'wb') as f:
        f.write(image_widget.value)
    update_counts()

save_button.on_click(save_snapshot)
update_counts()

display(widgets.HBox([image_widget, widgets.VBox([class_select, save_button, count_label])]))

## 프로젝트 종료하기
데이터 수집이 완료되었으면 카메라를 정지하여 자원을 해제합니다.

In [ ]:
camera.unobserve_all()
camera.stop()
print('Camera stopped successfully!')